In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from sklearn.cluster import KMeans
import time


### Robot Sample Locations

In [ ]:
# Build robot data table 
ranges = pd.read_csv('range.csv')
locations = pd.read_csv('robot_locations.csv')

robots = (
    locations.merge(ranges, on='index', how='inner')
    .rename(columns={'index': 'robot_id'})
    [['robot_id', 'longitude', 'latitude', 'range']]
    .sort_values('robot_id')
    .reset_index(drop=True)
)

#robots.to_csv('robot_table.csv', index=False)
robots.head()

In [ ]:
print(f'Number of robots: {len(robots)}')
print(f'Longitude range: {robots.longitude.min():.3f} to {robots.longitude.max():.3f}')
print(f'Latitude range: {robots.latitude.min():.3f} to {robots.latitude.max():.3f}')
print(f'Robot range: {robots["range"].min():.3f} to {robots["range"].max():.3f}')

grid_size = 25
lon_edges = np.linspace(robots['longitude'].min(), robots['longitude'].max(), grid_size + 1)
lat_edges = np.linspace(robots['latitude'].min(), robots['latitude'].max(), grid_size + 1)

fig, ax = plt.subplots(figsize=(10, 8))
heatmap = ax.hist2d(
    robots['longitude'],
    robots['latitude'],
    bins=[lon_edges, lat_edges],
    cmap='YlGnBu',
    alpha=0.65
)
ax.scatter(
    robots['longitude'],
    robots['latitude'],
    s=18,
    c='crimson',
    edgecolors='white',
    linewidths=0.3,
    marker='o',
    alpha=0.95,
    label='Robot position'
)
plt.colorbar(heatmap[3], ax=ax, label='Robots per grid cell')
ax.set_title('Robot Locations on a Grid')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_xticks(lon_edges)
ax.set_yticks(lat_edges)
ax.tick_params(axis='x', labelrotation=90)
ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.4)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Random sample of robots

robots_subset = robots.sample(n=100, random_state=1).reset_index(drop=True).copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    robots_subset['longitude'],
    robots_subset['latitude'],
    c=robots_subset['range'],
    cmap='viridis',
    s=42,
    edgecolors='white',
    linewidths=0.4,
    alpha=0.95
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Robot range')
ax.set_title('Sampled Robot Locations Colored by Range')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.4)
plt.tight_layout()
plt.show()


### Model

In [ ]:
# Model parameters from the assignment.
planning_days = 365 # run model for a year to account for annual costs
m = 8
q = 2
c_b = 5000
c_h = 1000
c_m = 500
c_c = 0.42
lambda_ = 0.012
r_min = 10
r_max = 175


#### Location

Option 2 appears to be the better option for results

In [ ]:
### Initialise with weighted k-means clustering to find central locations for stations

coords = robots_subset[["longitude", "latitude"]].to_numpy() # coordinates for clustering
weights = 1.0 / np.maximum(robots_subset["range"].to_numpy(), 1e-6) # weight by inverse of range to prioritize closer robots

# k = number of stations to try for run
k_values = [27]
# k_values = range(2, 7)

stations_by_k = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=1, n_init=10)
    km.fit(coords, sample_weight=weights)

    stations = pd.DataFrame(
        km.cluster_centers_,
        columns=["longitude", "latitude"]
    )
    stations["station_id"] = range(k)
    stations["k"] = k
    stations = stations[["k", "station_id", "longitude", "latitude"]]

    stations_by_k[k] = stations

    print(f"k = {k}: created {len(stations)} initial stations")

display(stations_by_k[k_values[0]])


#### Allocation

Function allocates each robot to one candidate station, subject to the capacity constraints of the stations. 

1) Compute maximum robot capacity per station:
m * q

2) For every robot-station pair:
 - compute distance
 - check whether the robot can reach the station
 - compute charging cost
 - compute transport penalty if needed
 - compute total assignment cost

3) Store all robot-station assignment possibilities in one table.

4) Sort those possibilities by cheapest assignment cost.

5) Greedily assign robots:
 - if robot already assigned, skip
 - if station is full, skip
 - otherwise assign robot to that station

Outputs:
 - assignments_df: which robot is assigned to which station
 - station_load: number of assigned robots at each station

In [ ]:
def allocation_step(robots_subset, stations, planning_days, m, q, c_c, c_h, r_max):
    
    station_capacity = m * q # max number of robots a station can serve
    assignment_options = [] # list to store all possible robot-station assignment options

    for _, robot in robots_subset.iterrows(): # loop over every robot
        i = int(robot["robot_id"]) # robot ID
        r_i = float(robot["range"]) # robot's range
        robot_lon = float(robot["longitude"]) # robot's longitude
        robot_lat = float(robot["latitude"]) # robot's latitude

        for _, station in stations.iterrows(): # loop over every station 
            j = int(station["station_id"]) # station ID
            station_lon = float(station["longitude"]) # station longitude
            station_lat = float(station["latitude"]) # station latitude

            # robot-station euclidean distance
            d_ij = np.sqrt(
                (robot_lon - station_lon) ** 2 +
                (robot_lat - station_lat) ** 2
            )

            # binary variable indicating if robot i is unreachable from station j
            unreachable = int(d_ij > r_i)

            # Charging cost:
            # reachable   -> c_c * (r_max - r_i + d_ij)
            # unreachable -> c_c * (r_max - r_i)
            charging_cost = planning_days * c_c * (
                (r_max - r_i + d_ij) * (1 - unreachable)
                + (r_max - r_i) * unreachable
            )

            # Transport penalty if unreachable
            transport_cost = planning_days * c_h * unreachable

            total_cost = charging_cost + transport_cost

            assignment_options.append({
                "robot_id": i,
                "station_id": j,
                "distance": d_ij,
                "unreachable": unreachable,
                "assign_cost": total_cost
            })



    # Sort all assignment options by cost (lowest first)
    assignment_options_df = pd.DataFrame(assignment_options).sort_values(
        by="assign_cost"
    ).reset_index(drop=True)

    assigned_robots = set() #set of asigned robots
    station_load = {j: 0 for j in stations["station_id"]} # number of robots assigned to station j 
    assignments = [] # list to store assignments


    # Greedy assignment over all robot-station pairs, cheapest first
    for _, row in assignment_options_df.iterrows(): # loop over all assignment options sorted by cost
        i = int(row["robot_id"]) # robot ID
        j = int(row["station_id"]) # station ID

        # skip if robot is already assigned
        if i in assigned_robots: 
            continue

        # skip if station is at capacity
        if station_load[j] >= station_capacity:
            continue

        # accept assignment 
        assignments.append({
            "robot_id": i,
            "station_id": j,
            "distance": float(row["distance"]),
            "unreachable": int(row["unreachable"]),
            "assign_cost": float(row["assign_cost"])
        })

        assigned_robots.add(i) # robot i has now been assigned 
        station_load[j] += 1 # station j now has one more assigned robot 

    assignments_df = pd.DataFrame(assignments)

    if len(assignments_df) != len(robots_subset):
        raise RuntimeError(
            f"Only assigned {len(assignments_df)} of {len(robots_subset)} robots. Increase k or change the construction step."
        )

    return assignments_df, station_load


#### Location Update

At this stage:
 - robot assignments are fixed
 - station locations are changed

 - if a station is currently serving a certain group of robots then it should move toward those robots with extra pull from robots with smaller ranges (to avoid rescue penalties)
 
So this step tries to improve the station layout while holding assignments fixed.

In [ ]:
def location_update_step(robots_subset, stations, assignments_df):
    updated_stations = stations.copy()

    for j in updated_stations["station_id"]: # loop over every station j

        # extract robot ids of assigned robots for station j
        assigned_robot_ids = assignments_df.loc[
            assignments_df["station_id"] == j, "robot_id"
        ].tolist()

        # skip if station has no assigned robots
        if len(assigned_robot_ids) == 0:
            continue

        # create dataframe of only robots assigned to station j
        assigned_robots_df = robots_subset[
            robots_subset["robot_id"].isin(assigned_robot_ids)
        ].copy()

        # weights inversely proportional to robot range
        weights = 1.0 / np.maximum(assigned_robots_df["range"].to_numpy(), 1e-6)

        # define new location of station j (weighted average of assigned robot locations)
        new_longitude = np.average(assigned_robots_df["longitude"], weights=weights)
        new_latitude = np.average(assigned_robots_df["latitude"], weights=weights)

        updated_stations.loc[
            updated_stations["station_id"] == j, "longitude"
        ] = new_longitude

        updated_stations.loc[
            updated_stations["station_id"] == j, "latitude"
        ] = new_latitude

    return updated_stations


#### Evaluation

Function computes charger counts and total annual cost for the current solution.

1) Compute station loads:
Count how many robots are assigned to each station.

2) Compute chargers needed:
For each station:
z_j = ÃƒÂ¢Ã…â€™Ã‹â€ robots assigned / q ÃƒÂ¢Ã…â€™Ã¢â‚¬Â°.
This is the minimum number of chargers required.

3) Check feasibility:
Verify: z_j ÃƒÂ¢Ã¢â‚¬Â°Ã‚Â¤ m
for every station.
If any station needs more than m chargers, the current solution is infeasible.

4) Recompute assignment costs:
Because the station locations have just changed, distances must be recomputed.
For every assigned robot-station pair:
 - recompute d_ij
 - determine if unreachable
 - compute charging cost
 - compute transport cost
 - compute total assignment cost
 - E. Compute total annual cost

Total cost = station build cost + charger maintenance cost + charging cost + transport cost

In [ ]:
def evaluation_step(robots_subset, updated_stations, assignments_df, planning_days, q, m, c_b, c_m, c_c, c_h, r_max):

    # count number of robots assigned to each station
    station_loads_df = (
        assignments_df.groupby("station_id")
        .size()
        .reset_index(name="robots_assigned")
    )

    solution_stations = updated_stations.merge(
        station_loads_df,
        on="station_id",
        how="left"
    )

    # fill nan values
    solution_stations["robots_assigned"] = solution_stations["robots_assigned"].fillna(0).astype(int)

    # chargers needed at each station = ceil(robots_assigned / q)
    solution_stations["chargers"] = np.ceil(solution_stations["robots_assigned"] / q).astype(int)
    solution_stations["feasible_station"] = solution_stations["chargers"] <= m # check feasability

    all_robots_assigned = len(assignments_df) == len(robots_subset)
    solution_feasible = solution_stations["feasible_station"].all() and all_robots_assigned

    assignment_cost_records = [] # list to store updated assignment costs 

    # calculate new assignment costs based on upated station locations
    for _, row in assignments_df.iterrows():
        i = int(row["robot_id"])
        j = int(row["station_id"])

        robot = robots_subset[robots_subset["robot_id"] == i].iloc[0]
        station_row = solution_stations[solution_stations["station_id"] == j].iloc[0]

        r_i = float(robot["range"])

        d_ij = np.sqrt(
            (robot["longitude"] - station_row["longitude"]) ** 2 +
            (robot["latitude"] - station_row["latitude"]) ** 2
        )

        unreachable = int(d_ij > r_i)

        charging_cost = planning_days * c_c * (
            (r_max - r_i + d_ij) * (1 - unreachable)
            + (r_max - r_i) * unreachable
        )

        transport_cost = planning_days * c_h * unreachable
        total_cost = charging_cost + transport_cost

        assignment_cost_records.append({
            "robot_id": i,
            "station_id": j,
            "distance": d_ij,
            "unreachable": unreachable,
            "charging_cost": charging_cost,
            "transport_cost": transport_cost,
            "assign_cost": total_cost
        })

    solution_assignments = pd.DataFrame(assignment_cost_records)

    num_open_stations = (solution_stations["robots_assigned"] > 0).sum() # number of stations used
    fixed_cost = num_open_stations * c_b + solution_stations["chargers"].sum() * c_m # fixed costs 
    charging_cost_total = solution_assignments["charging_cost"].sum() # total charging costs
    transport_cost_total = solution_assignments["transport_cost"].sum() # total transport costs 
    total_solution_cost = fixed_cost + charging_cost_total + transport_cost_total # total cost of solution 

    return solution_stations, solution_assignments, total_solution_cost, solution_feasible


#### RUN HEURISTIC

This function runs the previous three steps: Allocation, Location Update, Evaluation   until convergence or a time-limit is reached

In [ ]:
def run_alternating_heuristic(
    robots_subset,
    initial_stations,
    planning_days,
    m,
    q,
    c_b,
    c_m,
    c_c,
    c_h,
    r_max,
    max_iter=30,
    tolerance=1e-4,
    verbose=True
):
    
    stations_current = initial_stations.copy()
    cost_history = [] # to track cost over iterations for convergence
    patience = 3
    small_improvement_count = 0

    for iteration in range(1, max_iter + 1):
        # Step: Allocation
        assignments_df, station_load = allocation_step(
            robots_subset, stations_current, planning_days, m, q, c_c, c_h, r_max
        )

        # Step: Location Update
        updated_stations = location_update_step(
            robots_subset, stations_current, assignments_df
        )

        # Step: Evaluation
        solution_stations, solution_assignments, total_solution_cost, solution_feasible = evaluation_step(
            robots_subset, updated_stations, assignments_df,
            planning_days, q, m, c_b, c_m, c_c, c_h, r_max
        )

        cost_history.append(total_solution_cost) # track cost

    
        if verbose:
            print(
                f"Iteration {iteration}: "
                f"cost = {total_solution_cost:,.2f}, "
                f"open stations = {(solution_stations['robots_assigned'] > 0).sum()}, "
                f"chargers = {solution_stations['chargers'].sum()}, "
                f"feasible = {solution_feasible}"
            )

        # Convergence check
        if iteration > 1:
            improvement = cost_history[-2] - cost_history[-1]

            if improvement >= 0 and improvement < tolerance:
                small_improvement_count += 1
            else:
                small_improvement_count = 0

            if small_improvement_count >= patience:
                if verbose:
                    print(f"Converged after {iteration} iterations.")
                break
       

        stations_current = updated_stations.copy()

    return {
        "final_stations": solution_stations,
        "final_assignments": solution_assignments,
        "final_total_cost": total_solution_cost,
        "cost_history": cost_history,
        "feasible": solution_feasible
    }


#### Run the Heuristic

In [ ]:
results_by_k = {}

for k in k_values:
    print(f"\nRunning alternating heuristic for k = {k}")
    
    results_by_k[k] = run_alternating_heuristic(
        robots_subset=robots_subset,
        initial_stations=stations_by_k[k],
        planning_days=planning_days,
        m=m,
        q=q,
        c_b=c_b,
        c_m=c_m,
        c_c=c_c,
        c_h=c_h,
        r_max=r_max,
        max_iter=30,
        tolerance=1e-4,
        verbose=False
    )

    print(f"k = {k}, final cost = {results_by_k[k]['final_total_cost']:,.2f}, feasible = {results_by_k[k]['feasible']}")


### Results

In [ ]:
summary_by_k = pd.DataFrame([
    {
        "k": k,
        "final_cost": results_by_k[k]["final_total_cost"],
        "feasible": results_by_k[k]["feasible"],
        "num_open_stations": int((results_by_k[k]["final_stations"]["robots_assigned"] > 0).sum()),
        "total_chargers": int(results_by_k[k]["final_stations"]["chargers"].sum()),
        "transport_robots": int(results_by_k[k]["final_assignments"]["unreachable"].sum())
    }
    for k in k_values
])

display(summary_by_k.sort_values("final_cost"))


In [ ]:
# Extract the best solution across all tested k values

best_k = summary_by_k.loc[summary_by_k["final_cost"].idxmin(), "k"]
best_result = results_by_k[best_k]

best_stations_df = best_result["final_stations"].copy()
best_assignments_df = best_result["final_assignments"].copy()
best_total_cost = best_result["final_total_cost"]
best_cost_history = best_result["cost_history"]

print("Best k:", best_k)
print("Best total cost:", best_total_cost)
print("Feasible:", best_result["feasible"])

display(best_stations_df)
display(best_assignments_df.head(20))


In [ ]:
best_k = summary_by_k.loc[summary_by_k["final_cost"].idxmin(), "k"]
best_result = results_by_k[best_k]

best_stations_df = best_result["final_stations"].copy()
best_assignments_df = best_result["final_assignments"].copy()
best_total_cost = best_result["final_total_cost"]
best_cost_history = best_result["cost_history"]


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(summary_by_k["k"], summary_by_k["final_cost"], marker="o")
plt.xlabel("k")
plt.ylabel("Final Total Cost")
plt.title("Final Heuristic Cost by Number of Stations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(best_cost_history) + 1), best_cost_history, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Total Cost")
plt.title(f"Convergence of Alternating Heuristic (k = {best_k})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

transport_robot_ids = set(
    best_assignments_df.loc[best_assignments_df["unreachable"] == 1, "robot_id"]
)

reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

# Plot robots
ax.scatter(
    reachable_robots["longitude"],
    reachable_robots["latitude"],
    c="red",
    s=40,
    label="Reachable robots"
)

if not transport_robots.empty:
    ax.scatter(
        transport_robots["longitude"],
        transport_robots["latitude"],
        c="darkorange",
        s=55,
        label="Transport-needed robots"
    )

# Plot final stations
ax.scatter(
    best_stations_df["longitude"],
    best_stations_df["latitude"],
    c="blue",
    s=140,
    marker="X",
    label="Stations"
)

# Label stations with ID and chargers
for _, row in best_stations_df.iterrows():
    ax.text(
        row["longitude"],
        row["latitude"],
        f"S{int(row['station_id'])}\nC={int(row['chargers'])}",
        fontsize=8,
        ha="left",
        va="bottom",
        color="black"
    )

# Plot assignment lines
for _, row in best_assignments_df.iterrows():
    robot = robots_subset[robots_subset["robot_id"] == row["robot_id"]].iloc[0]
    station = best_stations_df[best_stations_df["station_id"] == row["station_id"]].iloc[0]

    line_color = "darkorange" if row["unreachable"] == 1 else "gray"

    ax.plot(
        [robot["longitude"], station["longitude"]],
        [robot["latitude"], station["latitude"]],
        color=line_color,
        alpha=0.45,
        linewidth=1
    )

ax.set_title(f"Alternating Location-Allocation Heuristic\nBest k = {best_k}, Cost = {best_total_cost:,.2f}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
for k in summary_by_k["k"]:
    result = results_by_k[k]
    stations_df = result["final_stations"]
    assignments_df = result["final_assignments"]
    total_cost = result["final_total_cost"]

    fig, ax = plt.subplots(figsize=(10, 8))

    transport_robot_ids = set(
        assignments_df.loc[assignments_df["unreachable"] == 1, "robot_id"]
    )

    reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
    transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

    ax.scatter(reachable_robots["longitude"], reachable_robots["latitude"], c="red", s=35, label="Reachable robots")

    if not transport_robots.empty:
        ax.scatter(transport_robots["longitude"], transport_robots["latitude"], c="darkorange", s=50, label="Transport-needed robots")

    ax.scatter(stations_df["longitude"], stations_df["latitude"], c="blue", s=120, marker="X", label="Stations")

    for _, row in assignments_df.iterrows():
        robot = robots_subset[robots_subset["robot_id"] == row["robot_id"]].iloc[0]
        station = stations_df[stations_df["station_id"] == row["station_id"]].iloc[0]
        line_color = "darkorange" if row["unreachable"] == 1 else "gray"
        ax.plot(
            [robot["longitude"], station["longitude"]],
            [robot["latitude"], station["latitude"]],
            color=line_color,
            alpha=0.4,
            linewidth=1
        )

    ax.set_title(f"Alternating Heuristic Final Solution (k = {k})\nCost = {total_cost:,.2f}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


### Local Search

In [ ]:
# Start local search from the best alternating-heuristic solution
# results_by_k is keyed by the tested k values, not by row position.

start_k = best_k if "best_k" in globals() else k_values[0]
base_result = results_by_k[start_k]

current_solution = {
    "stations_df": base_result["final_stations"].copy(),
    "assignments_df": base_result["final_assignments"].copy(),
    "total_cost": base_result["final_total_cost"],
    "feasible": base_result["feasible"]
}

print("Starting local search from k:", start_k)
print("Starting cost:", f"{current_solution['total_cost']:,.2f}")

In [ ]:
def evaluate_station_layout(robots_subset, stations_df, planning_days, m, q, c_b, c_m, c_c, c_h, r_max):
    station_cols = ["station_id", "longitude", "latitude"]
    if "k" in stations_df.columns:
        station_cols = ["k"] + station_cols

    clean_stations = stations_df[station_cols].copy()

    assignments_df, _ = allocation_step(
        robots_subset, clean_stations, planning_days, m, q, c_c, c_h, r_max
    )

    solution_stations, solution_assignments, total_cost, feasible = evaluation_step(
        robots_subset, clean_stations, assignments_df,
        planning_days, q, m, c_b, c_m, c_c, c_h, r_max
    )

    return {
        "stations_df": solution_stations,
        "assignments_df": solution_assignments,
        "total_cost": total_cost,
        "feasible": feasible
    }


In [ ]:
### Propose moving each station slightly towards the centroid of its assigned robots

def generate_relocate_neighbours(current_solution, robots_subset, step_fraction=0.25):
    neighbours = []
    stations_df = current_solution["stations_df"]
    assignments_df = current_solution["assignments_df"]

    for _, station in stations_df.iterrows(): # for each current station
        j = int(station["station_id"])

        # get assigned robots 
        assigned_ids = assignments_df.loc[
            assignments_df["station_id"] == j, "robot_id"
        ].tolist()

        # skip if no assigned robots
        if len(assigned_ids) == 0:
            continue

        assigned_robots = robots_subset[robots_subset["robot_id"].isin(assigned_ids)].copy()

        # compute centroid
        target_lon = assigned_robots["longitude"].mean()
        target_lat = assigned_robots["latitude"].mean()

        new_stations = stations_df.copy()
        old_lon = float(station["longitude"])
        old_lat = float(station["latitude"])

        # propose move towards centroid from current location (some multiple of the distance between the current location and centroid)
        new_stations.loc[new_stations["station_id"] == j, "longitude"] = old_lon + step_fraction * (target_lon - old_lon)
        new_stations.loc[new_stations["station_id"] == j, "latitude"] = old_lat + step_fraction * (target_lat - old_lat)

        neighbours.append(("relocate", j, new_stations))

    return neighbours


In [ ]:
### Replace two low utilised stations with one between them

def generate_merge_neighbours(
    current_solution,
    m,
    q,
    max_pairs=10,
    max_distance=None, # Add this!!!!!

    max_utilisation=0.25
):
    neighbours = []

    stations_df = current_solution["stations_df"].copy()
    assignments_df = current_solution["assignments_df"].copy()

    stations_base = stations_df.drop(columns=["robots_assigned"], errors="ignore").copy()

    # get number of assignments to each station
    station_loads = (
        assignments_df.groupby("station_id")
        .size()
        .reset_index(name="robots_assigned")
    )

    stations_with_loads = stations_base.merge(
        station_loads,
        on="station_id",
        how="left"
    )

    # stations with no assigned robots get zero 
    stations_with_loads["robots_assigned"] = (
        stations_with_loads["robots_assigned"].fillna(0).astype(int)
    )

    # station capacity and utilisation
    station_capacity = m * q
    stations_with_loads["utilisation"] = (
        stations_with_loads["robots_assigned"] / station_capacity
    )

    # Only open stations
    open_stations = stations_with_loads[
        stations_with_loads["robots_assigned"] > 0
    ].copy()

    pair_rows = [] # for candidate pairs
    rows = open_stations.to_dict("records")

    for a in range(len(rows)):
        for b in range(a + 1, len(rows)): # loop over every pair of open stations
            s1 = rows[a]
            s2 = rows[b]

            # Both stations must be low-utilisation
            if s1["utilisation"] > max_utilisation or s2["utilisation"] > max_utilisation:
                continue

            # calculate distance between stations
            dist = np.sqrt(
                (s1["longitude"] - s2["longitude"]) ** 2 +
                (s1["latitude"] - s2["latitude"]) ** 2
            )

            # Optional distance limit
            if max_distance is not None and dist > max_distance:
                continue

            combined_load = int(s1["robots_assigned"]) + int(s2["robots_assigned"])

            pair_rows.append({
                "distance": dist,
                "combined_load": combined_load,
                "j1": int(s1["station_id"]),
                "j2": int(s2["station_id"])
            })

    # Prefer closest low-load pairs
    pair_rows = sorted(pair_rows, key=lambda x: (x["distance"], x["combined_load"])) # prefer closer pairs, then with lower combined load

    for pair in pair_rows[:max_pairs]:
        j1 = pair["j1"]
        j2 = pair["j2"]

        new_stations = stations_base.copy()

        s1 = new_stations[new_stations["station_id"] == j1].iloc[0]
        s2 = new_stations[new_stations["station_id"] == j2].iloc[0]

        # propose new replacement station at the centre of pair of stations
        merged_lon = 0.5 * (s1["longitude"] + s2["longitude"])
        merged_lat = 0.5 * (s1["latitude"] + s2["latitude"])

        new_stations.loc[new_stations["station_id"] == j1, "longitude"] = merged_lon
        new_stations.loc[new_stations["station_id"] == j1, "latitude"] = merged_lat
        new_stations = new_stations[new_stations["station_id"] != j2].reset_index(drop=True)

        neighbours.append(("merge", (j1, j2), new_stations))

    return neighbours


In [ ]:
### Propose splitting stations with expensive assignments

def generate_split_neighbours(
    current_solution,
    robots_subset,
    max_split_candidates=5,
    min_station_load=6
):
    neighbours = []

    stations_df = current_solution["stations_df"].copy()
    assignments_df = current_solution["assignments_df"].copy()

    stations_base = stations_df.drop(
        columns=["robots_assigned", "chargers", "feasible_station"],
        errors="ignore"
    ).copy()

    # Score stations by number of unreachable robots, average assignment cost, then total assignment cost
    station_scores = (
        assignments_df.groupby("station_id")
        .agg(
            robots_assigned=("robot_id", "count"),
            avg_assign_cost=("assign_cost", "mean"),
            total_assign_cost=("assign_cost", "sum"),
            unreachable_count=("unreachable", "sum")
        )
        .reset_index()
        .sort_values(
            ["unreachable_count", "avg_assign_cost", "total_assign_cost"], # 
            ascending=[False, False, False]
        )
    )

    # Only consider stations with enough robots to justify a split
    candidate_stations = station_scores[
        station_scores["robots_assigned"] >= min_station_load
    ].head(max_split_candidates)

    current_ids = stations_base["station_id"].astype(int).tolist()
    next_station_id = max(current_ids) + 1 if current_ids else 0

    for _, score_row in candidate_stations.iterrows():
        j = int(score_row["station_id"])

        assigned_ids = assignments_df.loc[
            assignments_df["station_id"] == j, "robot_id"
        ].tolist()

        assigned_robots = robots_subset[
            robots_subset["robot_id"].isin(assigned_ids)
        ].copy()

        if len(assigned_robots) < 2:
            continue

        coords = assigned_robots[["longitude", "latitude"]].to_numpy()
        weights = 1.0 / np.maximum(assigned_robots["range"].to_numpy(), 1e-6)

        # Use 2-means to split this station's robot cluster into two groups
        km = KMeans(n_clusters=2, random_state=1, n_init=10)
        km.fit(coords, sample_weight=weights)

        centers = km.cluster_centers_

        new_stations = stations_base.copy()

        # Reuse the old station_id for one child station
        new_stations.loc[new_stations["station_id"] == j, "longitude"] = float(centers[0, 0])
        new_stations.loc[new_stations["station_id"] == j, "latitude"] = float(centers[0, 1])

        # Add the second child station with a new id
        extra_station = {
            "station_id": next_station_id,
            "longitude": float(centers[1, 0]),
            "latitude": float(centers[1, 1]),
        }

        if "k" in new_stations.columns:
            extra_station["k"] = new_stations["k"].iloc[0]

        new_stations = pd.concat(
            [new_stations, pd.DataFrame([extra_station])],
            ignore_index=True
        )

        neighbours.append(("split", j, new_stations))
        next_station_id += 1

    return neighbours


In [ ]:
def generate_all_neighbours(current_solution, robots_subset, m, q):
    neighbours = []

    neighbours += generate_relocate_neighbours(
        current_solution,
        robots_subset,
        step_fraction=0.25
    )

    neighbours += generate_merge_neighbours(
        current_solution,
        m=m,
        q=q,
        max_pairs=10,
        max_distance=25,
        max_utilisation=0.50
    )

    neighbours += generate_split_neighbours(
        current_solution,
        robots_subset,
        max_split_candidates=5,
        min_station_load=6
    )

    return neighbours


In [ ]:
### Local Search combined function

def find_first_improving_location_allocation_neighbour(
    current_solution,
    robots_subset,
    planning_days,
    m,
    q,
    c_b,
    c_m,
    c_c,
    c_h,
    r_max,
    start_time=None,
    global_time_limit_seconds=None
):
    current_cost = current_solution["total_cost"]

    for move_type, move_info, candidate_stations in generate_all_neighbours(current_solution, robots_subset, m, q):
        if start_time is not None and global_time_limit_seconds is not None:
            if time.time() - start_time >= global_time_limit_seconds:
                print("Time limit reached during neighbour search.")
                return None, None

        candidate_solution = evaluate_station_layout(
            robots_subset,
            candidate_stations,
            planning_days,
            m,
            q,
            c_b,
            c_m,
            c_c,
            c_h,
            r_max
        )

        if not candidate_solution["feasible"]:
            continue

        if candidate_solution["total_cost"] < current_cost:
            return (move_type, move_info), candidate_solution

    return None, None


In [ ]:
import time

ls_results_by_k = {}

global_time_limit_seconds = 300   # 5 minutes per k

for k in k_values:
    print(f"\nRunning local search for k = {k}")

    base_result = results_by_k[k]

    current_solution = {
        "stations_df": base_result["final_stations"].copy(),
        "assignments_df": base_result["final_assignments"].copy(),
        "total_cost": base_result["final_total_cost"],
        "feasible": base_result["feasible"]
    }

    start_time = time.time()
    local_search_history = [current_solution["total_cost"]]
    iteration = 0

    while True:
        elapsed = time.time() - start_time

        if elapsed >= global_time_limit_seconds:
            print(f"Time limit reached for k = {k}.")
            break

        iteration += 1
        print(f"k = {k}, local search iteration {iteration}, elapsed = {elapsed:.1f}s")

        move, improving_solution = find_first_improving_location_allocation_neighbour(
            current_solution=current_solution,
            robots_subset=robots_subset,
            planning_days=planning_days,
            m=m,
            q=q,
            c_b=c_b,
            c_m=c_m,
            c_c=c_c,
            c_h=c_h,
            r_max=r_max,
            start_time=start_time,
            global_time_limit_seconds=global_time_limit_seconds
        )

        if improving_solution is None:
            print(f"No improving neighbour found for k = {k}.")
            break

        print("Accepted move:", move)
        print("Old cost:", current_solution["total_cost"])
        print("New cost:", improving_solution["total_cost"])

        current_solution = improving_solution
        local_search_history.append(current_solution["total_cost"])

    ls_results_by_k[k] = {
        "final_stations": current_solution["stations_df"].copy(),
        "final_assignments": current_solution["assignments_df"].copy(),
        "final_total_cost": current_solution["total_cost"],
        "feasible": current_solution["feasible"],
        "cost_history": local_search_history,
        "elapsed_time": time.time() - start_time
    }


In [ ]:
ls_summary_by_k = pd.DataFrame([
    {
        "k": k,
        "initial_cost": results_by_k[k]["final_total_cost"],
        "final_cost": ls_results_by_k[k]["final_total_cost"],
        "improvement": results_by_k[k]["final_total_cost"] - ls_results_by_k[k]["final_total_cost"],
        "feasible": ls_results_by_k[k]["feasible"],
        "num_open_stations": int((ls_results_by_k[k]["final_stations"]["robots_assigned"] > 0).sum()),
        "total_chargers": int(ls_results_by_k[k]["final_stations"]["chargers"].sum()),
        "transport_robots": int(ls_results_by_k[k]["final_assignments"]["unreachable"].sum()),
        "elapsed_time": ls_results_by_k[k]["elapsed_time"]
    }
    for k in k_values
])

display(ls_summary_by_k.sort_values("final_cost"))


In [ ]:
best_k = ls_summary_by_k.loc[ls_summary_by_k["final_cost"].idxmin(), "k"]
best_result = ls_results_by_k[best_k]

best_stations_df = best_result["final_stations"].copy()
best_assignments_df = best_result["final_assignments"].copy()
best_total_cost = best_result["final_total_cost"]
best_cost_history = best_result["cost_history"]

print("Best k:", best_k)
print("Best total cost:", best_total_cost)
print("Feasible:", best_result["feasible"])

display(best_stations_df)
display(best_assignments_df.head(20))


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(ls_summary_by_k["k"], ls_summary_by_k["final_cost"], marker="o")
plt.xlabel("k")
plt.ylabel("Final Total Cost")
plt.title("Final Local Search Cost by Number of Stations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
ls_summary_by_k = pd.DataFrame([
    {
        "k": k,
        "final_cost": ls_results_by_k[k]["final_total_cost"],
        "feasible": ls_results_by_k[k]["feasible"],
        "num_open_stations": int((ls_results_by_k[k]["final_stations"]["robots_assigned"] > 0).sum()),
        "total_chargers": int(ls_results_by_k[k]["final_stations"]["chargers"].sum()),
        "transport_robots": int(ls_results_by_k[k]["final_assignments"]["unreachable"].sum())
    }
    for k in k_values
])

display(ls_summary_by_k.sort_values("final_cost"))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(best_cost_history) + 1), best_cost_history, marker="o")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Cost")
ax.set_title(f"Convergence of Alternating Heuristic (k = {best_k})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
#fig.savefig("Graphs/assignments_1c_b_local_search_1072.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

transport_robot_ids = set(
    best_assignments_df.loc[best_assignments_df["unreachable"] == 1, "robot_id"]
)

reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

ax.scatter(
    reachable_robots["longitude"],
    reachable_robots["latitude"],
    c="red",
    s=40,
    label="Reachable robots"
)

if not transport_robots.empty:
    ax.scatter(
        transport_robots["longitude"],
        transport_robots["latitude"],
        c="darkorange",
        s=55,
        label="Transport-needed robots"
    )

ax.scatter(
    best_stations_df["longitude"],
    best_stations_df["latitude"],
    c="blue",
    s=140,
    marker="X",
    label="Stations"
)

for _, row in best_assignments_df.iterrows():
    robot = robots_subset[robots_subset["robot_id"] == row["robot_id"]].iloc[0]
    station = best_stations_df[best_stations_df["station_id"] == row["station_id"]].iloc[0]

    line_color = "darkorange" if row["unreachable"] == 1 else "gray"

    ax.plot(
        [robot["longitude"], station["longitude"]],
        [robot["latitude"], station["latitude"]],
        color=line_color,
        alpha=0.45,
        linewidth=1
    )

ax.set_title(f"Best Local Search Solution\nBest k = {best_k}, Cost = {best_total_cost:,.2f}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
'''
for k in k_values:
    result = ls_results_by_k[k]
    stations_df = result["final_stations"]
    assignments_df = result["final_assignments"]
    total_cost = result["final_total_cost"]

    fig, ax = plt.subplots(figsize=(10, 8))

    transport_robot_ids = set(
        assignments_df.loc[assignments_df["unreachable"] == 1, "robot_id"]
    )

    reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
    transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

    ax.scatter(
        reachable_robots["longitude"],
        reachable_robots["latitude"],
        c="red",
        s=35,
        label="Reachable robots"
    )

    if not transport_robots.empty:
        ax.scatter(
            transport_robots["longitude"],
            transport_robots["latitude"],
            c="darkorange",
            s=50,
            label="Transport-needed robots"
        )

    ax.scatter(
        stations_df["longitude"],
        stations_df["latitude"],
        c="blue",
        s=120,
        marker="X",
        label="Stations"
    )

    for _, row in assignments_df.iterrows():
        robot = robots_subset[robots_subset["robot_id"] == row["robot_id"]].iloc[0]
        station = stations_df[stations_df["station_id"] == row["station_id"]].iloc[0]

        line_color = "darkorange" if row["unreachable"] == 1 else "gray"

        ax.plot(
            [robot["longitude"], station["longitude"]],
            [robot["latitude"], station["latitude"]],
            color=line_color,
            alpha=0.4,
            linewidth=1
        )

    ax.set_title(f"Final Local Search Solution (k = {k})\nCost = {total_cost:,.2f}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
'''

In [ ]:
# Summary metrics for heuristic vs local-search results

comparison_k = best_k
heuristic_summary_result = results_by_k[comparison_k]
local_search_summary_result = ls_results_by_k[comparison_k]

heuristic_summary_stations = heuristic_summary_result["final_stations"].copy()
heuristic_summary_assignments = heuristic_summary_result["final_assignments"].copy()

local_search_summary_stations = local_search_summary_result["final_stations"].copy()
local_search_summary_assignments = local_search_summary_result["final_assignments"].copy()

heuristic_summary_buildings_cost = (
    int((heuristic_summary_stations["robots_assigned"] > 0).sum()) * c_b
    + heuristic_summary_stations["chargers"].sum() * c_m
)
heuristic_summary_assignment_cost = heuristic_summary_assignments["charging_cost"].sum()
heuristic_summary_transport_cost = heuristic_summary_assignments["transport_cost"].sum()

local_search_summary_buildings_cost = (
    int((local_search_summary_stations["robots_assigned"] > 0).sum()) * c_b
    + local_search_summary_stations["chargers"].sum() * c_m
)
local_search_summary_assignment_cost = local_search_summary_assignments["charging_cost"].sum()
local_search_summary_transport_cost = local_search_summary_assignments["transport_cost"].sum()

heuristic_vs_local_search_summary = pd.DataFrame([
    {
        "Stage": "Post heuristic",
        "k": comparison_k,
        "Number of robots": len(robots_subset),
        "Number of stations": int((heuristic_summary_stations["robots_assigned"] > 0).sum()),
        "Number of chargers": int(heuristic_summary_stations["chargers"].sum()),
        "Total cost": heuristic_summary_result["final_total_cost"],
        "Assignment cost": heuristic_summary_assignment_cost,
        "Buildings cost": heuristic_summary_buildings_cost,
        "Transport cost": heuristic_summary_transport_cost,
        "Robots needing transport": int(heuristic_summary_assignments["unreachable"].sum()),
        "MIP gap": "N/A",
        "MIP gap (%)": "N/A",
    },
    {
        "Stage": "Post local search",
        "k": comparison_k,
        "Number of robots": len(robots_subset),
        "Number of stations": int((local_search_summary_stations["robots_assigned"] > 0).sum()),
        "Number of chargers": int(local_search_summary_stations["chargers"].sum()),
        "Total cost": local_search_summary_result["final_total_cost"],
        "Assignment cost": local_search_summary_assignment_cost,
        "Buildings cost": local_search_summary_buildings_cost,
        "Transport cost": local_search_summary_transport_cost,
        "Robots needing transport": int(local_search_summary_assignments["unreachable"].sum()),
        "MIP gap": "N/A",
        "MIP gap (%)": "N/A",
    },
])

display(
    heuristic_vs_local_search_summary.style.format({
        "k": "{:,.0f}",
        "Number of robots": "{:,.0f}",
        "Number of stations": "{:,.0f}",
        "Number of chargers": "{:,.0f}",
        "Total cost": "{:,.2f}",
        "Assignment cost": "{:,.2f}",
        "Buildings cost": "{:,.2f}",
        "Transport cost": "{:,.2f}",
        "Robots needing transport": "{:,.0f}",
    })
)

In [ ]:
# Compare assignments after the heuristic and after local search for the best k

comparison_k = best_k
heuristic_compare_result = results_by_k[comparison_k]
local_search_compare_result = ls_results_by_k[comparison_k]

heuristic_compare_stations = heuristic_compare_result["final_stations"].copy()
heuristic_compare_assignments = heuristic_compare_result["final_assignments"].copy()
heuristic_compare_cost = heuristic_compare_result["final_total_cost"]

local_search_compare_stations = local_search_compare_result["final_stations"].copy()
local_search_compare_assignments = local_search_compare_result["final_assignments"].copy()
local_search_compare_cost = local_search_compare_result["final_total_cost"]


def _plot_assignment_comparison_panel(ax, stations_df, assignments_df, title):
    station_key = "station_id" if "station_id" in stations_df.columns else "candidate_id"
    assignment_key = "station_id" if "station_id" in assignments_df.columns else "candidate_id"

    transport_robot_ids = set(
        assignments_df.loc[assignments_df["unreachable"] == 1, "robot_id"]
    )

    reachable_robots = robots_subset[~robots_subset["robot_id"].isin(transport_robot_ids)]
    transport_robots = robots_subset[robots_subset["robot_id"].isin(transport_robot_ids)]

    ax.scatter(
        reachable_robots["longitude"],
        reachable_robots["latitude"],
        c="red",
        s=35,
        alpha=0.75,
        label="Reachable robots"
    )

    if not transport_robots.empty:
        ax.scatter(
            transport_robots["longitude"],
            transport_robots["latitude"],
            c="darkorange",
            s=50,
            alpha=0.85,
            label="Transport-needed robots"
        )

    ax.scatter(
        stations_df["longitude"],
        stations_df["latitude"],
        c="blue",
        s=125,
        marker="X",
        label="Stations"
    )

    station_lookup = stations_df.set_index(station_key)
    robot_lookup = robots_subset.set_index("robot_id")

    for _, row in assignments_df.iterrows():
        robot_id = row["robot_id"]
        station_id = row[assignment_key]

        if robot_id not in robot_lookup.index or station_id not in station_lookup.index:
            continue

        robot = robot_lookup.loc[robot_id]
        station = station_lookup.loc[station_id]
        line_color = "darkorange" if row["unreachable"] == 1 else "gray"

        ax.plot(
            [robot["longitude"], station["longitude"]],
            [robot["latitude"], station["latitude"]],
            color=line_color,
            alpha=0.38,
            linewidth=1
        )

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True)

_plot_assignment_comparison_panel(
    axes[0],
    heuristic_compare_stations,
    heuristic_compare_assignments,
    f"After heuristic (k = {comparison_k})\nCost = {heuristic_compare_cost:,.2f}"
)

_plot_assignment_comparison_panel(
    axes[1],
    local_search_compare_stations,
    local_search_compare_assignments,
    f"After local search (k = {comparison_k})\nCost = {local_search_compare_cost:,.2f}"
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle("Assignment Comparison: Heuristic vs Local Search", fontsize=14)
plt.xticks(rotation=90)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
#fig.savefig("Graphs/assignments_1c_b_heuristic_vs_local_search_1072.png", dpi=300, bbox_inches="tight")
plt.show()